# 01 - QC & Preprocessing

**Goal:** load the four samples, throw out low-quality cells, normalise, and pick the most informative genes.

**Why each step exists**
- *Filter cells:* the sequencer produces garbage 'cells' (empty droplets, doublets, dying cells). We remove them so they don't distort everything downstream.
- *Normalise:* different cells were sequenced to different depths. Normalisation makes their expression values comparable.
- *Variable genes:* most genes are noise. We keep the ~2000 that actually vary between cells, which is what carries biological signal.

**Deliverable note:** the rubric asks you to *justify thresholds*. Run the plots FIRST, look at where the bulk of cells sits, then set the numbers in `config.py` and explain them in the slides.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))   # make the src/ package importable
from src import config as cfg
from src import utils as ut
ut.setup_scanpy()

import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print("scanpy", sc.__version__)

## Load the four samples into one object

In [ ]:
adata = ut.load_all_samples()
adata.obs['sample'].value_counts()

## Compute QC metrics
Flag mitochondrial genes (human = names starting with `MT-`). A high fraction of mitochondrial reads is the classic signature of a stressed or dying cell.

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True, percent_top=None)
adata.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].describe()

## Look at the distributions BEFORE filtering
These plots are what you use to justify your thresholds. Adjust the values in `config.py` so the cut lines (red) sit at the edges of the main population, not through the middle of it.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sc.pl.violin(adata, 'n_genes_by_counts', jitter=0.3, ax=axes[0], show=False)
axes[0].axhline(cfg.QC_MIN_GENES_PER_CELL, color='r', ls='--')
axes[0].axhline(cfg.QC_MAX_GENES_PER_CELL, color='r', ls='--')
sc.pl.violin(adata, 'total_counts', jitter=0.3, ax=axes[1], show=False)
sc.pl.violin(adata, 'pct_counts_mt', jitter=0.3, ax=axes[2], show=False)
axes[2].axhline(cfg.QC_MAX_PCT_MITO, color='r', ls='--')
plt.tight_layout(); plt.savefig(cfg.FIG_DIR / '01_qc_violins.png', dpi=150); plt.show()

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')

## Apply the filters
Thresholds come from `config.py`. Cells outside the gates and genes seen in almost no cells are removed.

In [ ]:
n_before = adata.n_obs
sc.pp.filter_cells(adata, min_genes=cfg.QC_MIN_GENES_PER_CELL)
sc.pp.filter_genes(adata, min_cells=cfg.QC_MIN_CELLS_PER_GENE)
adata = adata[adata.obs['n_genes_by_counts'] < cfg.QC_MAX_GENES_PER_CELL].copy()
adata = adata[adata.obs['pct_counts_mt'] < cfg.QC_MAX_PCT_MITO].copy()
print(f'cells: {n_before} -> {adata.n_obs}')

## Normalise + log-transform
Standard scanpy recipe: scale every cell to 10,000 counts, then `log1p`. We stash a raw copy in `.raw` because differential expression later wants the un-scaled values.

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata

## Highly variable genes

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=cfg.N_TOP_GENES, flavor='seurat_v3',
                            batch_key=cfg.BATCH_KEY)
sc.pl.highly_variable_genes(adata)

## Save checkpoint for notebook 02

In [ ]:
ut.save_checkpoint(adata, '01_qc_done')